In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from mimiciii_db import DB
from mimiciii_db.config import db_url

- icu stays 24 hrs admission
- age >=16
- elective vs nonelective
- patients, admissions, icustays

labevents, chartevents

sofa:
- respiratory system PaO2/FiO2
- coagulation platelet count
- liver bilirubin level
- cardiovascular blood pressure
- central nervous system
- renal

each 4 points, sum of 24

In [8]:
db = DB.from_url(db_url())


In [22]:
icustays = db.table_df("icustays", schema="mimiciii")
icustays

,row_id,subject_id,hadm_id,icustay_id,dbsource,first_careunit,last_careunit,first_wardid,last_wardid,intime,outtime,los
0,365,268,110404,280836,carevue,MICU,MICU,52,52,2198-02-14 23:27:38,2198-02-18 05:26:11,3.2490
1,366,269,106296,206613,carevue,MICU,MICU,52,52,2170-11-05 11:05:29,2170-11-08 17:46:57,3.2788
2,367,270,188028,220345,carevue,CCU,CCU,57,57,2128-06-24 15:05:20,2128-06-27 12:32:29,2.8939
3,368,271,173727,249196,carevue,MICU,SICU,52,23,2120-08-07 23:12:42,2120-08-10 00:39:04,2.0600
4,369,272,164716,210407,carevue,CCU,CCU,57,57,2186-12-25 21:08:04,2186-12-27 12:01:13,1.6202
...,...,...,...,...,...,...,...,...,...,...,...,...
61527,59806,94944,143774,201233,metavision,CSRU,CSRU,15,15,2104-04-15 10:18:16,2104-04-17 14:51:00,2.1894
61528,59807,94950,123750,283653,metavision,CCU,CCU,7,7,2155-12-08 05:33:16,2155-12-10 17:24:58,2.4942
61529,59808,94953,196881,241585,metavision,SICU,SICU,57,57,2160-03-03 16:09:11,2160-03-04 14:22:33,0.9259
61530,59809,94954,118475,202802,metavision,CSRU,CSRU,15,15,2183-03-25 09:53:10,2183-03-27 17:55:03,2.3346


In [ ]:
patients = db.table_df("patients", schema="mimiciii")
patients.head()

,row_id,subject_id,gender,dob,dod,dod_hosp,dod_ssn,expire_flag
0,234,249,F,2075-03-13,NaT,NaT,NaT,0
1,235,250,F,2164-12-27,2188-11-22,2188-11-22,NaT,1
2,236,251,M,2090-03-15,NaT,NaT,NaT,0
3,237,252,M,2078-03-06,NaT,NaT,NaT,0
4,238,253,F,2089-11-26,NaT,NaT,NaT,0


In [ ]:
admissions = db.table_df("admissions", schema="mimiciii")
admissions.head()

,row_id,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,religion,marital_status,ethnicity,edregtime,edouttime,diagnosis,hospital_expire_flag,has_chartevents_data
0,21,22,165315,2196-04-09 12:26:00,2196-04-10 15:54:00,NaT,EMERGENCY,EMERGENCY ROOM ADMIT,DISC-TRAN CANCER/CHLDRN H,Private,None,UNOBTAINABLE,MARRIED,WHITE,2196-04-09 10:06:00,2196-04-09 13:24:00,BENZODIAZEPINE OVERDOSE,0,1
1,22,23,152223,2153-09-03 07:15:00,2153-09-08 19:10:00,NaT,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,None,CATHOLIC,MARRIED,WHITE,NaT,NaT,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1
2,23,23,124321,2157-10-18 19:34:00,2157-10-25 14:00:00,NaT,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,NaT,NaT,BRAIN MASS,0,1
3,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaT,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,None,PROTESTANT QUAKER,SINGLE,WHITE,NaT,NaT,INTERIOR MYOCARDIAL INFARCTION,0,1
4,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaT,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,None,UNOBTAINABLE,MARRIED,WHITE,2160-11-02 01:01:00,2160-11-02 04:27:00,ACUTE CORONARY SYNDROME,0,1


In [9]:
# set your schema name here
SCHEMA = "mimiciii"

from sqlalchemy import text

def exec_sql(db, sql: str, params=None):
    """Run DDL/CTE/INSERT… statements that don't return a DataFrame."""
    with db.engine.begin() as conn:
        conn.execute(text(sql), params or {})


In [12]:
exec_sql(db, f"""
DROP TABLE IF EXISTS tmp_first_icu;
CREATE TEMP TABLE tmp_first_icu AS
WITH first_icu AS (
  SELECT
    icustay_id,
    hadm_id,
    subject_id,
    intime,
    ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY intime) AS rn
  FROM {SCHEMA}.icustays
),
first_only AS (
  SELECT
    icustay_id,
    hadm_id,
    subject_id,
    intime
  FROM first_icu
  WHERE rn = 1
),
adult AS (
  SELECT
    f.icustay_id,
    f.hadm_id,
    f.subject_id,
    f.intime,
    adm.admittime,
    pat.dob,
    EXTRACT(EPOCH FROM (adm.admittime - pat.dob))/86400.0/365.2425 AS age
  FROM first_only f
  JOIN {SCHEMA}.admissions adm
    ON adm.hadm_id = f.hadm_id
  JOIN {SCHEMA}.patients pat
    ON pat.subject_id = f.subject_id
)
SELECT *
FROM adult
WHERE age >= 16;
""")


In [15]:
db.query_df("SELECT COUNT(*) AS n, MIN(age) AS min_age, MAX(age) AS max_age FROM tmp_first_icu")


,n,min_age,max_age
0,38597,16.016324,306.8892


In [16]:
exec_sql(db, f"""
DROP TABLE IF EXISTS tmp_ce24;
CREATE TEMP TABLE tmp_ce24 AS
SELECT ce.icustay_id, ce.charttime, ce.itemid, ce.valuenum
FROM {SCHEMA}.chartevents ce
JOIN tmp_first_icu fi USING (icustay_id)
WHERE ce.charttime>=fi.intime
  AND ce.charttime< fi.intime + INTERVAL '24 hours'
  AND ce.valuenum IS NOT NULL
  AND ce.error IS DISTINCT FROM 1;

DROP TABLE IF EXISTS tmp_le24;
CREATE TEMP TABLE tmp_le24 AS
SELECT fi.icustay_id, le.charttime, le.itemid, le.valuenum
FROM {SCHEMA}.labevents le
JOIN tmp_first_icu fi USING (hadm_id)
WHERE le.charttime>=fi.intime
  AND le.charttime< fi.intime + INTERVAL '24 hours'
  AND le.valuenum IS NOT NULL
  AND le.flag IS DISTINCT FROM 'ERROR';

DROP TABLE IF EXISTS tmp_oe24;
CREATE TEMP TABLE tmp_oe24 AS
SELECT fi.icustay_id, oe.charttime, oe.value AS urine_cc
FROM {SCHEMA}.outputevents oe
JOIN tmp_first_icu fi USING (icustay_id, hadm_id)
WHERE oe.charttime>=fi.intime
  AND oe.charttime< fi.intime + INTERVAL '24 hours'
  AND oe.value IS NOT NULL;
""")

# row counts to sanity check
db.query_df("""
SELECT
  (SELECT COUNT(*) FROM tmp_ce24) AS ce_rows,
  (SELECT COUNT(*) FROM tmp_le24) AS le_rows,
  (SELECT COUNT(*) FROM tmp_oe24) AS oe_rows
""")


,ce_rows,le_rows,oe_rows
0,23121069,2833208,790635
